In [ ]:

from pathlib import Path
from google.colab import drive
import pandas as pd
import shutil
import os

# --------------------------------------------------
# 1. Mount the correct Google Drive
# --------------------------------------------------
candidates = [
    "/content/drive5",
    "/content/drive6",
    "/content/drive7",
    "/content/drive8",
    "/content/drive9",
]

MOUNT = None

for candidate in candidates:
    p = Path(candidate)

    # Reuse a mount only if the expected project is visible.
    existing_project = p / "MyDrive/Ocular_Project"
    if existing_project.is_dir():
        MOUNT = p
        print("Using accessible project:", existing_project)
        break

    # Otherwise choose a clean mount point.
    if not p.exists() or not any(p.iterdir()):
        MOUNT = p
        drive.mount(str(MOUNT))
        break

if MOUNT is None:
    raise RuntimeError(
        "No suitable mount point found. Inspect /content/drive*."
    )

PROJECT = MOUNT / "MyDrive/Ocular_Project"

# --------------------------------------------------
# 2. Verify Notebook 04 outputs
# --------------------------------------------------
SPLIT_CSV = (
    PROJECT / "results/IITD/notebook04_baseline_recognition/"
    "IITD_baseline_dataset_split.csv"
)
META_CSV = (
    PROJECT / "results/IITD/notebook02_preprocessing/"
    "IITD_preprocessed_metadata.csv"
)

assert PROJECT.is_dir(), f"Project folder missing: {PROJECT}"
assert SPLIT_CSV.is_file(), f"Baseline split missing: {SPLIT_CSV}"
assert META_CSV.is_file(), f"Preprocessing metadata missing: {META_CSV}"

split_df = pd.read_csv(SPLIT_CSV)
meta_df = pd.read_csv(META_CSV)

print("\nDrive:", MOUNT)
print("Project:", PROJECT)
print("Split columns:", split_df.columns.tolist())
print("Metadata columns:", meta_df.columns.tolist())
print("\nSplit counts:")
print(split_df["split"].value_counts())


In [2]:
from pathlib import Path

cache = Path("/content/IITD_traditional_aug_cache")

print("Cache exists:", cache.exists())

if cache.exists():
    files = list(cache.rglob("*.png"))
    print("PNG images copied:", len(files))
    print("Manifest exists:", (cache / "training_manifest.csv").exists())

Cache exists: True
PNG images copied: 954
Manifest exists: False


In [3]:

from pathlib import Path
import pandas as pd
import shutil
from tqdm.auto import tqdm

LOCAL_CACHE = Path("/content/IITD_traditional_aug_cache")
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

# Reuse the dataframe and source paths from the earlier setup cell.
assert "train_df" in globals(), "Run the setup cell first."
assert "processed_path" in train_df.columns, "Missing source paths."
assert "original_relative_path" in train_df.columns, "Missing relative paths."

local_paths = []

for row in tqdm(
    train_df.itertuples(index=False),
    total=len(train_df),
    desc="Caching training images"
):
    relative = Path(row.original_relative_path)
    destination = LOCAL_CACHE / relative.with_suffix(".png")
    destination.parent.mkdir(parents=True, exist_ok=True)

    # Skip files already copied successfully.
    if not destination.is_file():
        shutil.copy2(row.processed_path, destination)

    local_paths.append(str(destination))

train_df["local_path"] = local_paths

# Create the manifest only after all copies finish.
manifest_path = LOCAL_CACHE / "training_manifest.csv"
train_df.to_csv(manifest_path, index=False)

cached_count = sum(Path(p).is_file() for p in local_paths)

print("\nLOCAL CACHE VERIFIED")
print("Expected images:", len(train_df))
print("Accessible cached images:", cached_count)
print("Manifest exists:", manifest_path.exists())
print("Cache folder:", LOCAL_CACHE)

assert cached_count == len(train_df), "Some images are still missing."
assert manifest_path.exists(), "Manifest was not created."
print("All training images are ready for augmentation!")


Caching training images:   0%|          | 0/1526 [00:00<?, ?it/s]


LOCAL CACHE VERIFIED
Expected images: 1526
Accessible cached images: 1526
Manifest exists: True
Cache folder: /content/IITD_traditional_aug_cache
All training images are ready for augmentation!


In [5]:

from pathlib import Path
from PIL import Image
import shutil
from tqdm.auto import tqdm

LOCAL_CACHE = Path("/content/IITD_traditional_aug_cache")

bad_files = []

for row in tqdm(train_df.itertuples(index=False),
                total=len(train_df),
                desc="Checking cached images"):
    cached = Path(row.local_path)

    try:
        with Image.open(cached) as im:
            im.verify()
    except Exception:
        bad_files.append((cached, Path(row.processed_path)))

print("Invalid cached images:", len(bad_files))

for cached, source in tqdm(bad_files, desc="Repairing cache"):
    if not source.is_file():
        raise FileNotFoundError(f"Drive source missing: {source}")

    cached.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, cached)

    # Verify the repaired copy.
    with Image.open(cached) as im:
        im.verify()

print("Repair complete. Repaired files:", len(bad_files))


Checking cached images:   0%|          | 0/1526 [00:00<?, ?it/s]

Invalid cached images: 1


Repairing cache:   0%|          | 0/1 [00:00<?, ?it/s]

Repair complete. Repaired files: 1


In [6]:

from pathlib import Path
from PIL import Image, ImageEnhance
import pandas as pd
import random
import json
import time
from tqdm.auto import tqdm

# -------------------- Configuration --------------------
SEED = 42
AUGMENTATIONS_PER_IMAGE = 3

random.seed(SEED)
rng = random.Random(SEED)

PROJECT = Path("/content/drive5/MyDrive/Ocular_Project")
LOCAL_CACHE = Path("/content/IITD_traditional_aug_cache")
AUG_LOCAL = Path("/content/IITD_traditional_augmented_run03")
AUG_LOCAL.mkdir(parents=True, exist_ok=True)

OUT_DIR = PROJECT / "results/IITD/notebook05_traditional_augmentation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(LOCAL_CACHE / "training_manifest.csv")

assert len(train_df) == 1526, f"Unexpected training rows: {len(train_df)}"
assert train_df["local_path"].map(lambda p: Path(p).is_file()).all(), (
    "Some local images are missing. Re-run the caching cell."
)

# -------------------- Augmentation function --------------------
def augment_image(image, rng):
    image = image.rotate(
        rng.uniform(-8, 8),
        resample=Image.Resampling.BILINEAR,
        fillcolor=0
    )

    image = ImageEnhance.Brightness(image).enhance(
        rng.uniform(0.85, 1.15)
    )
    image = ImageEnhance.Contrast(image).enhance(
        rng.uniform(0.85, 1.15)
    )

    dx = int(rng.uniform(-0.03, 0.03) * image.width)
    dy = int(rng.uniform(-0.03, 0.03) * image.height)

    image = image.transform(
        image.size,
        Image.Transform.AFFINE,
        (1, 0, -dx, 0, 1, -dy),
        resample=Image.Resampling.BILINEAR,
        fillcolor=0
    )
    return image

# -------------------- Generate images --------------------
records = []
start = time.time()

for row in tqdm(
    train_df.itertuples(index=False),
    total=len(train_df),
    desc="Generating augmented images"
):
    with Image.open(row.local_path) as im:
        original = im.convert("L")

    label_name = str(row.label_name)
    subject = str(row.subject)
    eye = str(row.eye)

    class_dir = AUG_LOCAL / label_name
    class_dir.mkdir(parents=True, exist_ok=True)

    safe_stem = Path(row.original_relative_path).with_suffix("").as_posix()
    safe_stem = safe_stem.replace("/", "_")

    for aug_id in range(1, AUGMENTATIONS_PER_IMAGE + 1):
        augmented = augment_image(original.copy(), rng)
        filename = f"{safe_stem}_aug{aug_id}.png"
        output_path = class_dir / filename
        augmented.save(output_path, format="PNG")

        records.append({
            "original_relative_path": row.original_relative_path,
            "augmented_local_path": str(output_path),
            "filename": filename,
            "subject": subject,
            "eye": eye,
            "label_name": label_name,
            "label": int(row.label),
            "split": "train",
            "augmentation_id": aug_id,
            "augmentation_methods":
                "rotation_translation_brightness_contrast"
        })

aug_df = pd.DataFrame(records)

assert len(aug_df) == len(train_df) * AUGMENTATIONS_PER_IMAGE
assert aug_df["augmented_local_path"].map(
    lambda p: Path(p).is_file()
).all()

# -------------------- Save metadata and ZIP --------------------
metadata_path = OUT_DIR / "IITD_traditional_augmented_metadata.csv"
aug_df.to_csv(metadata_path, index=False)

zip_path = OUT_DIR / "IITD_traditional_augmented_images.zip"

import zipfile
with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    for row in tqdm(
        aug_df.itertuples(index=False),
        total=len(aug_df),
        desc="Packaging ZIP"
    ):
        image_path = Path(row.augmented_local_path)
        archive_name = (
            Path("train") / row.label_name / image_path.name
        )
        zf.write(image_path, arcname=archive_name.as_posix())

summary = {
    "dataset": "IIT Delhi Iris Database",
    "source_training_images": len(train_df),
    "augmentations_per_image": AUGMENTATIONS_PER_IMAGE,
    "generated_augmented_images": len(aug_df),
    "training_examples_including_originals":
        len(train_df) + len(aug_df),
    "validation_and_test_augmented": False,
    "image_size": "224x224",
    "image_mode": "grayscale",
    "methods": [
        "small rotation",
        "small translation",
        "brightness variation",
        "contrast variation"
    ],
    "seed": SEED,
    "runtime_seconds": round(time.time() - start, 2),
    "archive": zip_path.name
}

summary_path = OUT_DIR / "IITD_traditional_augmentation_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\nNOTEBOOK 05 RESULTS")
print(json.dumps(summary, indent=2))
print("\nFiles saved:")
for p in [metadata_path, zip_path, summary_path]:
    print(f"{p.name}: {p.stat().st_size / (1024**2):.2f} MB")


Generating augmented images:   0%|          | 0/1526 [00:00<?, ?it/s]

Packaging ZIP:   0%|          | 0/4578 [00:00<?, ?it/s]


NOTEBOOK 05 RESULTS
{
  "dataset": "IIT Delhi Iris Database",
  "source_training_images": 1526,
  "augmentations_per_image": 3,
  "generated_augmented_images": 4578,
  "training_examples_including_originals": 6104,
  "validation_and_test_augmented": false,
  "image_size": "224x224",
  "image_mode": "grayscale",
  "methods": [
    "small rotation",
    "small translation",
    "brightness variation",
    "contrast variation"
  ],
  "seed": 42,
  "runtime_seconds": 45.03,
  "archive": "IITD_traditional_augmented_images.zip"
}

Files saved:
IITD_traditional_augmented_metadata.csv: 0.70 MB
IITD_traditional_augmented_images.zip: 123.46 MB
IITD_traditional_augmentation_summary.json: 0.00 MB


## Conclusion

In this notebook, traditional image augmentation was performed on the training portion of the IIT Delhi Iris Database to prepare additional ocular images for subsequent recognition experiments.

A total of **1,526 original training images** were used to generate **4,578 augmented images**, with three augmented versions produced per source image. The transformations included small rotations, translations, brightness variations, and contrast variations. The generated images were maintained in grayscale format at 224 × 224 pixels.

The resulting training collection contains 6,104 examples when the original training images are counted alongside their augmented versions. Validation and test images were not augmented, preserving the original evaluation data.

### Generated Outputs

The following artifacts were saved:

- `IITD_traditional_augmented_metadata.csv` — metadata describing the generated images and their source images.
- `IITD_traditional_augmented_images.zip` — compressed archive containing the augmented images.
- `IITD_traditional_augmentation_summary.json` — summary of the augmentation configuration and output counts.

The generation and packaging process completed successfully in approximately 45 seconds.

### Limitations

This notebook generates augmented images but does not train or evaluate a recognition model. Therefore, the effect of traditional augmentation on recognition accuracy has not yet been established. The transformations may introduce useful variations, but their effectiveness must be measured experimentally.

### Next Steps

The generated dataset will support subsequent experiments in the project. Recognition performance should be evaluated using the established dataset split, with the original baseline serving as the reference for comparisons.